# DeepStego — M2 en Kaggle

Activa **GPU** e **Internet**. Adjunta el dataset `deepstego-div2k` y el dataset privado
creado con `outputs/deepstego-kaggle.zip`. Ejecuta las celdas en orden.
La primera ejecución entrena una época; después puedes continuar hasta 20.

Esta CNN es una versión inicial. Las métricas de aceptación y la lectura de
los QR deben comprobarse después del entrenamiento.


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

projects = [
    path.parent
    for path in INPUT_ROOT.rglob("pyproject.toml")
    if (path.parent / "src/deepstego/train.py").is_file()
]
if not projects:
    archives = [
        path
        for path in INPUT_ROOT.rglob("*")
        if path.is_file() and path.name.casefold() == "deepstego-kaggle.zip"
    ]
    if not archives:
        raise RuntimeError(
            "Falta el código: adjunta deepstego-kaggle.zip como otro dataset, "
            "además de DeepStego-DIV2K. El notebook solo no contiene las redes."
        )
    if len(archives) > 1:
        raise RuntimeError(f"Hay varios ZIP de código; deja uno adjunto: {archives}")
    with ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT / "source-upload")
    projects = [
        path.parent
        for path in (WORK_ROOT / "source-upload").rglob("pyproject.toml")
        if (path.parent / "src/deepstego/train.py").is_file()
    ]
if len(projects) != 1:
    raise RuntimeError(f"Se esperaba una única copia del código: {projects}")

PROJECT_DIR = WORK_ROOT / "deepstego-code"
shutil.copytree(projects[0], PROJECT_DIR, dirs_exist_ok=True)
uv = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
if not Path(uv).is_file():
    installer = WORK_ROOT / "install-uv.sh"
    subprocess.run(
        ["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", str(installer)], check=True
    )
    subprocess.run(["sh", str(installer)], check=True)
    uv = str(Path.home() / ".local/bin/uv")
subprocess.run([uv, "pip", "install", "--python", sys.executable, "qrcode[pil]"], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Activa un acelerador GPU en la configuración del notebook.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## Datos

El notebook localiza las carpetas de imágenes dentro de `/kaggle/input`,
sin depender del nombre del dataset ni de las carpetas intermedias.
Para DIV2K completo, verifica 800 imágenes de entrenamiento y 100 de validación.


In [ ]:
from deepstego.data import DIV2KDataset, find_div2k_split

TRAIN_DIR = find_div2k_split(INPUT_ROOT, "DIV2K_train_HR")
VAL_DIR = find_div2k_split(INPUT_ROOT, "DIV2K_valid_HR")

train_data = DIV2KDataset(TRAIN_DIR, training=True, seed=42)
val_data = DIV2KDataset(VAL_DIR, training=False, seed=43)
print("Entrenamiento:", len(train_data), TRAIN_DIR)
print("Validación:", len(val_data), VAL_DIR)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index in range(4):
    sample = train_data[index % len(train_data)]
    axes[0, index].imshow(sample["cover"].permute(1, 2, 0))
    axes[1, index].imshow(sample["secret"][0], cmap="gray", vmin=0, vmax=1)
    axes[1, index].set_title(sample["text"], fontsize=8)
for axis in axes.flat:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Entrenar

Primero ejecuta una época. Para continuar, cambia `EPOCHS=20` y
`RESUME="/kaggle/working/m2/last.pt"`. Las épocas son el total deseado.
Para iniciar otro experimento usa `RESUME=None` y una carpeta de salida nueva.

En otra sesión, adjunta los outputs anteriores y señala el `last.pt` adjunto.
Mantén `best.pt` a su lado y conserva los datos e hiperparámetros originales.


In [ ]:
from deepstego.train import TrainConfig, train

EPOCHS = 1
RESUME = None
OUTPUT_DIR = WORK_ROOT / "m2"

config = TrainConfig(
    train_dir=str(TRAIN_DIR),
    val_dir=str(VAL_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    batch_size=16,
    learning_rate=1e-3,
    beta=1.0,
    width=32,
    crops_per_image=4,
    num_workers=2,
    seed=42,
    device="cuda",
    resume=RESUME,
)
history = train(config)

## Revisar resultados

`best.pt` se selecciona por pérdida total de validación. El MSE del QR y el
PSNR ayudan a seguir el entrenamiento, pero no sustituyen SSIM ni la lectura
del mensaje. El ejemplo siguiente revela desde un PNG guardado y recargado.


In [ ]:
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for split in ("train", "val"):
    axes[0].plot(epochs, [row[split]["loss"] for row in history], "o-", label=split)
    axes[1].plot(epochs, [row[split]["secret_mse"] for row in history], "o-", label=split)
    axes[2].plot(epochs, [row[split]["psnr"] for row in history], "o-", label=split)
for axis, title in zip(axes, ("Pérdida total", "MSE del QR", "PSNR (dB)")):
    axis.set_title(title)
    axis.set_xlabel("Época")
    axis.legend()
    axis.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "curves.png")
plt.show()

In [ ]:
import numpy as np
from PIL import Image

from deepstego.models import HideNet, RevealNet, quantize_8bit

device = torch.device(config.device)
checkpoint = torch.load(OUTPUT_DIR / "best.pt", map_location=device, weights_only=True)
hide = HideNet(checkpoint["config"]["width"]).to(device)
reveal = RevealNet(checkpoint["config"]["width"]).to(device)
hide.load_state_dict(checkpoint["hide"])
reveal.load_state_dict(checkpoint["reveal"])
hide.eval()
reveal.eval()

sample = val_data[0]
cover = sample["cover"].unsqueeze(0).to(device)
secret = sample["secret"].unsqueeze(0).to(device)
with torch.inference_mode():
    container = quantize_8bit(hide(cover, secret))
    pixels = container[0].permute(1, 2, 0).mul(255).round().byte().cpu().numpy()
    Image.fromarray(pixels).save(OUTPUT_DIR / "container.png")
    with Image.open(OUTPUT_DIR / "container.png") as png:
        loaded = torch.from_numpy(np.array(png.convert("RGB"))).permute(2, 0, 1).float() / 255
    recovered = reveal(loaded.unsqueeze(0).to(device))[0, 0].cpu()

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
images = [sample["cover"].permute(1, 2, 0), sample["secret"][0], pixels, recovered]
for axis, image, title in zip(axes, images, ("Portadora", "QR original", "PNG", "QR revelado")):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
    axis.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "examples.png")
plt.show()
print("Texto esperado:", sample["text"])
print("Mejor época:", checkpoint["epoch"])
print("Outputs:", OUTPUT_DIR)

## Conservar el entrenamiento

Guarda los outputs con una versión del notebook o descarga `last.pt`, `best.pt`
y `history.json`. Los checkpoints se escriben al terminar cada época.
**Save & Run All** vuelve a ejecutar todas las celdas en otra sesión; revisa
`RESUME` antes de hacerlo.

Referencias: [Kaggle](https://www.kaggle.com/docs/notebooks) y
[uv](https://docs.astral.sh/uv/pip/environments/).
